# Graph Transformer for Molecular Graph Classification

This notebook trains small transformer-style models for graph-level classification on `ogbg-molhiv`. Each molecule is represented as a graph: atoms are node tokens, chemical bonds are edges, and the target is whether the molecule inhibits HIV replication.

The experiment compares three pooling variants of the same graph-transformer backbone:

1. Laplacian positional encoding + bond/adjacency attention bias + mean pooling
2. Laplacian positional encoding + bond/adjacency attention bias + max pooling
3. Laplacian positional encoding + bond/adjacency attention bias + mean+max pooling

The best epoch is selected by validation ROC-AUC, and the test ROC-AUC is reported only after choosing that epoch.

## How to Run

For a quick local check, keep `QUICK_RUN = True`. This uses fewer graphs, fewer batches, and two epochs.

For Colab/GPU training:

1. Upload this notebook to Colab.
2. Select a GPU runtime.
3. Set `INSTALL_PACKAGES = True` if `ogb` or `torch_geometric` are missing.
4. Keep `SAVE_OUTPUTS_TO_GOOGLE_DRIVE = True`.
5. Set `QUICK_RUN = False` for full training.
6. Run all cells.

When running in Colab, the notebook mounts Google Drive and creates this designated output folder:

```text
/content/drive/MyDrive/ModernStatisticsFinalProject/transformers
```

Inside it, the notebook saves:

- `data/graph_transformer_results.csv`
- `data/graph_transformer_history.csv`
- `plots/*.png`
- `checkpoints/*.pt`
- `cache/*.pt` for Laplacian positional encodings
- `ogb/` for the downloaded OGB dataset

When running outside Colab, the notebook saves to the local project folders under `FinalProject/data/transformers/` and `FinalProject/plots/transformers/`.


## Setup and Optional Installs

In [ ]:
# Set this to True in Colab if OGB / PyTorch Geometric are not installed.
INSTALL_PACKAGES = False

if INSTALL_PACKAGES:
    import subprocess
    import sys

    packages = [
        'ogb',
        'torch-geometric',
        'tqdm',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)


In [ ]:
import copy
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from ogb.graphproppred import Evaluator, PygGraphPropPredDataset
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# PyTorch 2.6 changed torch.load to weights_only=True by default.
# Some PyG/OGB processed dataset files contain PyG graph metadata objects,
# so we explicitly keep the older behavior for trusted local OGB/cache files.
try:
    from torch.serialization import add_safe_globals
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage
    add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
except Exception as exc:
    print('PyG safe-global registration skipped:', exc)

_original_torch_load = torch.load


def torch_load_compat(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)


torch.load = torch_load_compat

try:
    CODE_DIR = Path(__file__).resolve().parent
except NameError:
    CODE_DIR = Path.cwd()

PROJECT_DIR = CODE_DIR.parent if CODE_DIR.name.lower() == 'code' else CODE_DIR

SAVE_OUTPUTS_TO_GOOGLE_DRIVE = True
GOOGLE_DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/ModernStatisticsFinalProject/transformers')


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if SAVE_OUTPUTS_TO_GOOGLE_DRIVE and running_in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = GOOGLE_DRIVE_OUTPUT_DIR
else:
    OUTPUT_ROOT = PROJECT_DIR

DATA_DIR = OUTPUT_ROOT / 'data'
PLOTS_DIR = OUTPUT_ROOT / 'plots'
TRANSFORMER_DATA_DIR = DATA_DIR
TRANSFORMER_PLOTS_DIR = PLOTS_DIR
CACHE_DIR = OUTPUT_ROOT / 'cache'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
OGB_ROOT = OUTPUT_ROOT / 'ogb'

for path in [TRANSFORMER_DATA_DIR, TRANSFORMER_PLOTS_DIR, CACHE_DIR, CHECKPOINT_DIR, OGB_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('Output root:', OUTPUT_ROOT)
print('Stats folder:', TRANSFORMER_DATA_DIR)
print('Plots folder:', TRANSFORMER_PLOTS_DIR)
print('Checkpoint folder:', CHECKPOINT_DIR)


## Configuration

In [ ]:
SEED = 42
QUICK_RUN = True

BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-5

D_MODEL = 64
NUM_LAYERS = 3
NUM_HEADS = 4
FFN_DIM = 128
DROPOUT = 0.10
LAP_PE_DIM = 8
MAX_NODES = None

QUICK_EPOCHS = 2
QUICK_TRAIN_GRAPHS = 512
QUICK_VALID_GRAPHS = 256
QUICK_TEST_GRAPHS = 256
QUICK_MAX_TRAIN_BATCHES = 8
QUICK_MAX_EVAL_BATCHES = 4

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print('Device:', DEVICE)
print('Quick run:', QUICK_RUN)


## Load ogbg-molhiv and Inspect the Data

In [ ]:
dataset = PygGraphPropPredDataset(name='ogbg-molhiv', root=str(OGB_ROOT))
evaluator = Evaluator(name='ogbg-molhiv')
split_idx = dataset.get_idx_split()

train_idx = split_idx['train']
valid_idx = split_idx['valid']
test_idx = split_idx['test']

if QUICK_RUN:
    train_idx = train_idx[:QUICK_TRAIN_GRAPHS]
    valid_idx = valid_idx[:QUICK_VALID_GRAPHS]
    test_idx = test_idx[:QUICK_TEST_GRAPHS]

example = dataset[0]
print('Number of graphs:', len(dataset))
print('Train / valid / test:', len(train_idx), len(valid_idx), len(test_idx))
print('Example graph:', example)
print('Node feature shape:', tuple(example.x.shape))
print('Edge index shape:', tuple(example.edge_index.shape))
print('Edge feature shape:', tuple(example.edge_attr.shape))
print('Label shape:', tuple(example.y.shape))


## Feature Cardinalities

The node and edge features in `ogbg-molhiv` are categorical. We compute cardinalities from the full dataset and use embedding tables instead of treating these features as continuous numbers.

In [ ]:
def compute_feature_cardinalities(dataset):
    node_max = None
    edge_max = None

    for data in tqdm(dataset, desc='Scanning feature cardinalities'):
        x = data.x.long()
        edge_attr = data.edge_attr.long()

        cur_node_max = x.max(dim=0).values if x.numel() else torch.zeros(9, dtype=torch.long)
        cur_edge_max = edge_attr.max(dim=0).values if edge_attr.numel() else torch.zeros(3, dtype=torch.long)

        node_max = cur_node_max if node_max is None else torch.maximum(node_max, cur_node_max)
        edge_max = cur_edge_max if edge_max is None else torch.maximum(edge_max, cur_edge_max)

    node_cardinalities = (node_max + 1).tolist()
    edge_cardinalities = (edge_max + 1).tolist()
    return node_cardinalities, edge_cardinalities


node_cardinalities, edge_cardinalities = compute_feature_cardinalities(dataset)
print('Node feature cardinalities:', node_cardinalities)
print('Edge feature cardinalities:', edge_cardinalities)


## Laplacian Positional Encoding

We use the normalized graph Laplacian `L = I - D^{-1/2} A D^{-1/2}`. The positional encoding is built from the smallest nontrivial eigenvectors. For deterministic signs, each eigenvector is flipped so that its largest-magnitude entry is positive.

In [ ]:
def compute_laplacian_pe(edge_index, num_nodes, pe_dim):
    if num_nodes <= 0:
        return torch.zeros((0, pe_dim), dtype=torch.float32)

    device = edge_index.device
    adjacency = torch.zeros((num_nodes, num_nodes), dtype=torch.float32, device=device)
    if edge_index.numel() > 0:
        src, dst = edge_index[0], edge_index[1]
        adjacency[src, dst] = 1.0
        adjacency[dst, src] = 1.0

    degree = adjacency.sum(dim=1)
    inv_sqrt_degree = torch.zeros_like(degree)
    positive_degree = degree > 0
    inv_sqrt_degree[positive_degree] = degree[positive_degree].pow(-0.5)

    normalized_adjacency = inv_sqrt_degree[:, None] * adjacency * inv_sqrt_degree[None, :]
    laplacian = torch.eye(num_nodes, dtype=torch.float32, device=device) - normalized_adjacency

    eigenvalues, eigenvectors = torch.linalg.eigh(laplacian)
    del eigenvalues

    if num_nodes > 1:
        pe = eigenvectors[:, 1:1 + pe_dim]
    else:
        pe = eigenvectors[:, :0]

    if pe.shape[1] < pe_dim:
        padding = torch.zeros((num_nodes, pe_dim - pe.shape[1]), dtype=torch.float32, device=device)
        pe = torch.cat([pe, padding], dim=1)

    for j in range(pe.shape[1]):
        col = pe[:, j]
        if col.numel() and col[col.abs().argmax()] < 0:
            pe[:, j] = -col

    return pe.cpu().float()


class LaplacianPECache:
    def __init__(self, dataset, pe_dim, cache_path):
        self.dataset = dataset
        self.pe_dim = pe_dim
        self.cache_path = Path(cache_path)
        if self.cache_path.exists():
            self.cache = torch.load(self.cache_path, map_location='cpu')
        else:
            self.cache = {}

    def get(self, idx):
        idx = int(idx)
        if idx not in self.cache:
            data = self.dataset[idx]
            self.cache[idx] = compute_laplacian_pe(data.edge_index, data.num_nodes, self.pe_dim)
        return self.cache[idx]

    def save(self):
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.cache, self.cache_path)

    def precompute(self, indices):
        for idx in tqdm(indices, desc='Precomputing Laplacian PE'):
            self.get(int(idx))
        self.save()


pe_cache = LaplacianPECache(dataset, LAP_PE_DIM, CACHE_DIR / f'lap_pe_dim_{LAP_PE_DIM}.pt')
pe_cache.precompute(torch.cat([train_idx, valid_idx, test_idx]).tolist())


## Dataset Wrapper and Padded Collate

In [ ]:
class IndexedGraphDataset(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = [int(i) for i in indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        idx = self.indices[position]
        return self.dataset[idx], idx


def collate_graphs(batch):
    graphs, indices = zip(*batch)
    batch_size = len(graphs)
    node_counts = [int(g.num_nodes) for g in graphs]
    max_nodes = max(node_counts)
    if MAX_NODES is not None:
        max_nodes = min(max_nodes, MAX_NODES)

    node_feat_dim = graphs[0].x.shape[1]
    edge_feat_dim = graphs[0].edge_attr.shape[1]

    x_cat = torch.zeros((batch_size, max_nodes, node_feat_dim), dtype=torch.long)
    y = torch.zeros((batch_size, 1), dtype=torch.float32)
    node_mask = torch.zeros((batch_size, max_nodes), dtype=torch.bool)
    lap_pe = torch.zeros((batch_size, max_nodes, LAP_PE_DIM), dtype=torch.float32)
    edge_attr_dense = torch.zeros((batch_size, max_nodes, max_nodes, edge_feat_dim), dtype=torch.long)
    edge_mask = torch.zeros((batch_size, max_nodes, max_nodes), dtype=torch.bool)
    adjacency_mask = torch.zeros((batch_size, max_nodes, max_nodes), dtype=torch.bool)

    for b, (graph, idx) in enumerate(zip(graphs, indices)):
        n = min(int(graph.num_nodes), max_nodes)
        x_cat[b, :n] = graph.x[:n].long()
        y[b] = graph.y.view(1).float()
        node_mask[b, :n] = True
        lap = pe_cache.get(idx)
        lap_pe[b, :n] = lap[:n]

        edge_index = graph.edge_index.long()
        edge_attr = graph.edge_attr.long()
        if edge_index.numel() > 0:
            src, dst = edge_index[0], edge_index[1]
            keep = (src < n) & (dst < n)
            src, dst = src[keep], dst[keep]
            kept_attr = edge_attr[keep]
            edge_attr_dense[b, src, dst] = kept_attr
            edge_mask[b, src, dst] = True
            adjacency_mask[b, src, dst] = True
            adjacency_mask[b, dst, src] = True

    return {
        'x_cat': x_cat,
        'y': y,
        'node_mask': node_mask,
        'lap_pe': lap_pe,
        'edge_attr_dense': edge_attr_dense,
        'edge_mask': edge_mask,
        'adjacency_mask': adjacency_mask,
        'graph_idx': torch.tensor(indices, dtype=torch.long),
    }


train_loader = DataLoader(IndexedGraphDataset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_graphs)
valid_loader = DataLoader(IndexedGraphDataset(dataset, valid_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_graphs)
test_loader = DataLoader(IndexedGraphDataset(dataset, test_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_graphs)

batch = next(iter(train_loader))
for key, value in batch.items():
    print(key, tuple(value.shape) if hasattr(value, 'shape') else type(value))


## Model Components

In [ ]:
class CategoricalNodeEncoder(nn.Module):
    def __init__(self, cardinalities, d_model):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(int(card), d_model) for card in cardinalities])

    def forward(self, x_cat):
        out = 0
        for j, emb in enumerate(self.embeddings):
            out = out + emb(x_cat[:, :, j])
        return out


class LaplacianPEEncoder(nn.Module):
    def __init__(self, pe_dim, d_model):
        super().__init__()
        self.linear = nn.Linear(pe_dim, d_model)

    def forward(self, lap_pe):
        return self.linear(lap_pe)


class EdgeAttentionBias(nn.Module):
    def __init__(self, edge_cardinalities, num_heads):
        super().__init__()
        self.edge_embeddings = nn.ModuleList([nn.Embedding(int(card), num_heads) for card in edge_cardinalities])
        self.adjacency_bias = nn.Parameter(torch.zeros(num_heads))

    def forward(self, edge_attr_dense, edge_mask, adjacency_mask):
        bias = 0
        for j, emb in enumerate(self.edge_embeddings):
            bias = bias + emb(edge_attr_dense[:, :, :, j])
        bias = bias.permute(0, 3, 1, 2)
        bias = bias * edge_mask.unsqueeze(1).float()
        bias = bias + self.adjacency_bias.view(1, -1, 1, 1) * adjacency_mask.unsqueeze(1).float()
        return bias


class GraphMultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, node_mask, attention_bias):
        bsz, num_nodes, _ = x.shape
        q = self.q_proj(x).view(bsz, num_nodes, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(bsz, num_nodes, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(bsz, num_nodes, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores + attention_bias
        scores = scores.masked_fill(~node_mask[:, None, None, :], torch.finfo(scores.dtype).min)

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(bsz, num_nodes, self.d_model)
        out = self.out_proj(out)
        out = out * node_mask.unsqueeze(-1).float()
        return out


class GraphTransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ffn_dim, dropout):
        super().__init__()
        self.attention = GraphMultiHeadSelfAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, node_mask, attention_bias):
        x = self.norm1(x + self.dropout(self.attention(x, node_mask, attention_bias)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        x = x * node_mask.unsqueeze(-1).float()
        return x


class GraphTransformerClassifier(nn.Module):
    def __init__(self, node_cardinalities, edge_cardinalities, d_model, num_layers, num_heads, ffn_dim, dropout, lap_pe_dim, pool_type):
        super().__init__()
        assert pool_type in {'mean', 'max', 'mean_max'}
        self.pool_type = pool_type
        self.node_encoder = CategoricalNodeEncoder(node_cardinalities, d_model)
        self.pe_encoder = LaplacianPEEncoder(lap_pe_dim, d_model)
        self.edge_bias = EdgeAttentionBias(edge_cardinalities, num_heads)
        self.blocks = nn.ModuleList([
            GraphTransformerBlock(d_model, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])
        pooled_dim = d_model * 2 if pool_type == 'mean_max' else d_model
        self.head = nn.Sequential(
            nn.LayerNorm(pooled_dim),
            nn.Linear(pooled_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def pool(self, x, node_mask):
        mask = node_mask.unsqueeze(-1)
        count = mask.sum(dim=1).clamp(min=1)
        mean_pool = (x * mask.float()).sum(dim=1) / count

        x_for_max = x.masked_fill(~mask, torch.finfo(x.dtype).min)
        max_pool = x_for_max.max(dim=1).values

        if self.pool_type == 'mean':
            return mean_pool
        if self.pool_type == 'max':
            return max_pool
        return torch.cat([mean_pool, max_pool], dim=-1)

    def forward(self, batch):
        x = self.node_encoder(batch['x_cat']) + self.pe_encoder(batch['lap_pe'])
        x = x * batch['node_mask'].unsqueeze(-1).float()
        attention_bias = self.edge_bias(batch['edge_attr_dense'], batch['edge_mask'], batch['adjacency_mask'])

        for block in self.blocks:
            x = block(x, batch['node_mask'], attention_bias)

        graph_repr = self.pool(x, batch['node_mask'])
        return self.head(graph_repr)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Training and Evaluation Utilities

In [ ]:
def move_batch_to_device(batch, device):
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}


def labels_mask(y):
    return ~torch.isnan(y.view(-1))


def batch_accuracy(logits, y):
    mask = labels_mask(y)
    if int(mask.sum()) == 0:
        return np.nan, 0
    probs = torch.sigmoid(logits.view(-1)[mask])
    preds = (probs >= 0.5).long()
    target = y.view(-1)[mask].long()
    return float((preds == target).float().mean().item()), int(mask.sum())


def roc_auc_from_arrays(y_true, y_pred):
    if len(np.unique(y_true)) < 2:
        return np.nan
    try:
        return float(evaluator.eval({'y_true': y_true.reshape(-1, 1), 'y_pred': y_pred.reshape(-1, 1)})['rocauc'])
    except ValueError:
        return np.nan


def train_one_epoch(model, loader, optimizer, criterion, device, max_batches=None):
    model.train()
    total_loss = 0.0
    total_examples = 0
    acc_weighted_sum = 0.0
    y_true_all, y_pred_all = [], []

    for batch_idx, batch in enumerate(tqdm(loader, desc='train', leave=False)):
        if max_batches is not None and batch_idx >= max_batches:
            break
        batch = move_batch_to_device(batch, device)
        y = batch['y']
        mask = labels_mask(y)
        if int(mask.sum()) == 0:
            continue

        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits.view(-1)[mask], y.view(-1)[mask])
        loss.backward()
        optimizer.step()

        n = int(mask.sum())
        total_loss += float(loss.item()) * n
        total_examples += n
        acc, acc_n = batch_accuracy(logits.detach(), y)
        acc_weighted_sum += acc * acc_n

        y_true_all.append(y.view(-1)[mask].detach().cpu().numpy())
        y_pred_all.append(torch.sigmoid(logits.view(-1)[mask]).detach().cpu().numpy())

    y_true = np.concatenate(y_true_all) if y_true_all else np.array([])
    y_pred = np.concatenate(y_pred_all) if y_pred_all else np.array([])
    return {
        'loss': total_loss / max(total_examples, 1),
        'accuracy': acc_weighted_sum / max(total_examples, 1),
        'rocauc': roc_auc_from_arrays(y_true, y_pred) if len(y_true) else np.nan,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device, max_batches=None, desc='eval'):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    acc_weighted_sum = 0.0
    y_true_all, y_pred_all = [], []

    for batch_idx, batch in enumerate(tqdm(loader, desc=desc, leave=False)):
        if max_batches is not None and batch_idx >= max_batches:
            break
        batch = move_batch_to_device(batch, device)
        y = batch['y']
        mask = labels_mask(y)
        if int(mask.sum()) == 0:
            continue

        logits = model(batch)
        loss = criterion(logits.view(-1)[mask], y.view(-1)[mask])

        n = int(mask.sum())
        total_loss += float(loss.item()) * n
        total_examples += n
        acc, acc_n = batch_accuracy(logits, y)
        acc_weighted_sum += acc * acc_n

        y_true_all.append(y.view(-1)[mask].detach().cpu().numpy())
        y_pred_all.append(torch.sigmoid(logits.view(-1)[mask]).detach().cpu().numpy())

    y_true = np.concatenate(y_true_all) if y_true_all else np.array([])
    y_pred = np.concatenate(y_pred_all) if y_pred_all else np.array([])
    return {
        'loss': total_loss / max(total_examples, 1),
        'accuracy': acc_weighted_sum / max(total_examples, 1),
        'rocauc': roc_auc_from_arrays(y_true, y_pred) if len(y_true) else np.nan,
    }


## Experiment Runner

In [ ]:
def make_model(pool_type):
    model = GraphTransformerClassifier(
        node_cardinalities=node_cardinalities,
        edge_cardinalities=edge_cardinalities,
        d_model=D_MODEL,
        num_layers=NUM_LAYERS,
        num_heads=NUM_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        lap_pe_dim=LAP_PE_DIM,
        pool_type=pool_type,
    )
    return model.to(DEVICE)


def run_experiment(model_name, pool_type):
    set_seed(SEED)
    model = make_model(pool_type)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()

    epochs = QUICK_EPOCHS if QUICK_RUN else EPOCHS
    max_train_batches = QUICK_MAX_TRAIN_BATCHES if QUICK_RUN else None
    max_eval_batches = QUICK_MAX_EVAL_BATCHES if QUICK_RUN else None

    best_valid_rocauc = -np.inf
    best_epoch = None
    best_state = None
    history = []

    print()
    print(f'{model_name}: {count_parameters(model):,} trainable parameters')
    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, max_batches=max_train_batches)
        valid_metrics = evaluate(model, valid_loader, criterion, DEVICE, max_batches=max_eval_batches, desc='valid')

        row = {
            'model_name': model_name,
            'pool_type': pool_type,
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'valid_loss': valid_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'valid_accuracy': valid_metrics['accuracy'],
            'train_rocauc': train_metrics['rocauc'],
            'valid_rocauc': valid_metrics['rocauc'],
        }
        history.append(row)

        print(
            f"epoch {epoch:02d} | "
            f"train loss {row['train_loss']:.4f} acc {row['train_accuracy']:.4f} auc {row['train_rocauc']:.4f} | "
            f"valid loss {row['valid_loss']:.4f} acc {row['valid_accuracy']:.4f} auc {row['valid_rocauc']:.4f}"
        )

        if not np.isnan(valid_metrics['rocauc']) and valid_metrics['rocauc'] > best_valid_rocauc:
            best_valid_rocauc = valid_metrics['rocauc']
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, criterion, DEVICE, max_batches=max_eval_batches, desc='test')

    checkpoint_path = CHECKPOINT_DIR / f'{model_name}.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'best_epoch': best_epoch,
        'best_valid_rocauc': best_valid_rocauc,
        'pool_type': pool_type,
        'node_cardinalities': node_cardinalities,
        'edge_cardinalities': edge_cardinalities,
        'config': {
            'd_model': D_MODEL,
            'num_layers': NUM_LAYERS,
            'num_heads': NUM_HEADS,
            'ffn_dim': FFN_DIM,
            'dropout': DROPOUT,
            'lap_pe_dim': LAP_PE_DIM,
            'lr': LR,
            'weight_decay': WEIGHT_DECAY,
        },
    }, checkpoint_path)

    result = {
        'model_name': model_name,
        'pool_type': pool_type,
        'best_epoch': best_epoch,
        'best_valid_rocauc': best_valid_rocauc,
        'test_rocauc_at_best_valid': test_metrics['rocauc'],
        'final_train_loss': history[-1]['train_loss'],
        'final_valid_loss': history[-1]['valid_loss'],
        'parameter_count': count_parameters(model),
    }
    return pd.DataFrame(history), result


experiment_configs = [
    {'model_name': 'graph_transformer_mean_pool', 'pool_type': 'mean'},
    {'model_name': 'graph_transformer_max_pool', 'pool_type': 'max'},
    {'model_name': 'graph_transformer_mean_max_pool', 'pool_type': 'mean_max'},
]


In [ ]:
all_histories = []
all_results = []

for cfg in experiment_configs:
    history_df, result = run_experiment(**cfg)
    all_histories.append(history_df)
    all_results.append(result)
    pe_cache.save()

history = pd.concat(all_histories, ignore_index=True)
results = pd.DataFrame(all_results).sort_values('best_valid_rocauc', ascending=False).reset_index(drop=True)

history_path = TRANSFORMER_DATA_DIR / 'graph_transformer_history.csv'
results_path = TRANSFORMER_DATA_DIR / 'graph_transformer_results.csv'
history.to_csv(history_path, index=False)
results.to_csv(results_path, index=False)
print('Saved history to:', history_path)
print('Saved results to:', results_path)
print('Saved model weights under:', CHECKPOINT_DIR)
results


## Plots

In [ ]:
def plot_model_history(history, model_name):
    df = history[history['model_name'] == model_name]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df['epoch'], df['train_loss'], marker='o', label='train')
    ax.plot(df['epoch'], df['valid_loss'], marker='o', label='valid')
    ax.set_title(f'{model_name}: loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE loss')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(TRANSFORMER_PLOTS_DIR / f'{model_name}_loss.png', dpi=150)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df['epoch'], df['train_accuracy'], marker='o', label='train')
    ax.plot(df['epoch'], df['valid_accuracy'], marker='o', label='valid')
    ax.set_title(f'{model_name}: accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(TRANSFORMER_PLOTS_DIR / f'{model_name}_accuracy.png', dpi=150)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df['epoch'], df['valid_rocauc'], marker='o', color='tab:green')
    ax.set_title(f'{model_name}: validation ROC-AUC')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ROC-AUC')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(TRANSFORMER_PLOTS_DIR / f'{model_name}_valid_rocauc.png', dpi=150)
    plt.show()


for cfg in experiment_configs:
    plot_model_history(history, cfg['model_name'])


## Architecture Schematic

```text
Molecular graph
-> nodes as atom tokens
-> categorical atom embeddings
-> Laplacian positional encoding
-> graph transformer blocks with bond/adjacency attention bias
-> graph pooling: mean / max / mean+max
-> MLP classifier
-> HIV inhibition probability
```


## Final Result Table

In [ ]:
results[[
    'model_name',
    'pool_type',
    'best_epoch',
    'best_valid_rocauc',
    'test_rocauc_at_best_valid',
    'final_train_loss',
    'final_valid_loss',
    'parameter_count',
]]
